In [45]:
#data cleaning
#importing data
import pandas as pd
import numpy as np

raw_pilot_data = pd.read_csv(r"C:\Users\haran\metalab\109_cse\python_analysis\cse_109_pilot_data.csv")

#flagging practice trials
raw_pilot_data["block_marker"] = (
        raw_pilot_data["stimulus"]
        .str.contains(r"Blokk\s*1\s*kezdődik", case=False, regex=True, na=False)
    )
raw_pilot_data["start_seen"] = (
        raw_pilot_data.groupby("subj_code")["trial_index"]
        .transform(lambda x: (x == 0).cummax())
    )
raw_pilot_data["marker_seen"] = (
        raw_pilot_data.groupby("subj_code")["block_marker"]
        .transform(lambda x: x.cummax())
    )
raw_pilot_data["trial_exp"] = np.where(
        (raw_pilot_data["start_seen"] == 1) & (raw_pilot_data["marker_seen"] == 0),
        "practice",
        "experimental"
    )

#flagging previous trials
raw_pilot_data["prev_congruency"] = raw_pilot_data["congruency"].shift(4)
raw_pilot_data["prev_correct"] = raw_pilot_data["correct"].shift(4)

raw_pilot_data["correct"] = raw_pilot_data["correct"].map({"true": 1, "false": 0})
raw_pilot_data["prev_correct"] = raw_pilot_data["prev_correct"].map({"true": 1, "false": 0})

#flagging first trials
raw_pilot_data["first_trial"] = raw_pilot_data["stimulus"].shift(4).str.contains(r"Blokk \d+ kezdődik", regex = True)
raw_pilot_data["first_trial"] = raw_pilot_data["first_trial"].astype(int)

raw_pilot_data["rt"] = pd.to_numeric(raw_pilot_data["rt"], errors="coerce")

#filtering subjects with accuracy below <60%
accuracy = (
    raw_pilot_data[(raw_pilot_data["task"] == "probe") &
                   (raw_pilot_data["trial_exp"] != "practice") &
                   (raw_pilot_data["first_trial"] != 1)
                   ]
    .groupby("subj_code")["correct"]
    .mean()
    .reset_index(name="all_accuracy")
)

raw_pilot_data = (
    raw_pilot_data
    .merge(accuracy, on="subj_code", how="left")
)

raw_pilot_data = raw_pilot_data[
    raw_pilot_data["all_accuracy"] > 0.60
]

#filtering data
filtered_pilot_data = (
    raw_pilot_data[(raw_pilot_data["task"] == "probe") &
                   (raw_pilot_data["correct"] == 1) &
                   (raw_pilot_data["prev_correct"] == 1) &
                   (raw_pilot_data["trial_exp"] != "practice") &
                   (raw_pilot_data["first_trial"] != 1) &
                   (raw_pilot_data["rt"] >= 150)
    ]
)

processed_pilot_data = filtered_pilot_data[
    ["rt", "subj_code", "task", "congruency", "color", "monetary", "experiment", "prev_congruency", "all_accuracy"]
]